# Traductor de Lengua de Señas Mexicana (LSM)

**Alumno:** Axel Lopez  
**Institución:** CECyTEN Tepic  
**Módulo:** III — Inteligencia Artificial  
**Fecha:** Mayo 2026

---

## Descripción

Este proyecto detecta señas del abecedario de la Lengua de Señas Mexicana en tiempo real usando visión artificial y machine learning. La cámara captura la mano, MediaPipe extrae 21 puntos clave, se calculan los ángulos de cada dedo mediante la ley de cosenos, y un clasificador entrenado predice la letra correspondiente.

## Tecnologías utilizadas

- **Visión Artificial:** MediaPipe Tasks + OpenCV
- **Machine Learning:** scikit-learn (Random Forest / KNN)
- **PLN:** acumulación de letras para formar palabras
- **Lenguaje:** Python 3.13

In [ ]:
# Celda 1: Instalar dependencias
!pip install scikit-learn pandas numpy matplotlib seaborn joblib -q
print('Librerias instaladas correctamente')

In [ ]:
# Celda 2: Subir el dataset
# Sube el archivo dataset_señas.csv cuando aparezca el boton
from google.colab import files
uploaded = files.upload()
print('Archivo cargado:', list(uploaded.keys()))

In [ ]:
# Celda 3: Cargar y visualizar el dataset
# El dataset contiene 6 angulos por muestra (uno por dedo) y la letra correspondiente
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('dataset_señas.csv')

print('Total de muestras:', len(df))
print('Columnas:', list(df.columns))
print()
print('Muestras por letra:')
print(df['letra'].value_counts().sort_index().to_string())

df['letra'].value_counts().sort_index().plot(
    kind='bar', figsize=(12, 4), color='steelblue', edgecolor='black'
)
plt.title('Distribucion de muestras por letra')
plt.xlabel('Letra')
plt.ylabel('Cantidad de muestras')
plt.tight_layout()
plt.show()

In [ ]:
# Celda 4: Visualizar los angulos promedio por letra
# Cada letra tiene un patron de angulos caracteristico que la diferencia de las demas
import numpy as np
import seaborn as sns

angulos_cols = ['angle1', 'angle2', 'angle3', 'angle4', 'angle5', 'angle6']
promedios = df.groupby('letra')[angulos_cols].mean()

plt.figure(figsize=(14, 6))
sns.heatmap(promedios, annot=True, fmt='.0f', cmap='YlOrRd')
plt.title('Angulo promedio por dedo y letra')
plt.xlabel('Dedo (angle1=menique ... angle6=pulgar interno)')
plt.ylabel('Letra')
plt.tight_layout()
plt.show()

print('Cada fila muestra el patron de angulos que caracteriza a esa letra.')

In [ ]:
# Celda 5: Preparar datos y entrenar el modelo
# Se divide el dataset en 80% entrenamiento y 20% prueba
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X = df[angulos_cols].values
y = df['letra'].values

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

modelos = {
    'Random Forest':     RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN':               KNeighborsClassifier(n_neighbors=5),
    'Arbol de Decision': DecisionTreeClassifier(random_state=42),
}

print('Resultados por modelo:')
print('-' * 35)
resultados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    acc = accuracy_score(y_test, modelo.predict(X_test))
    resultados[nombre] = acc
    print(f'{nombre:25s}: {acc*100:.2f}%')

mejor_nombre = max(resultados, key=resultados.get)
mejor_modelo = modelos[mejor_nombre]
print(f'\nMejor modelo: {mejor_nombre} con {resultados[mejor_nombre]*100:.2f}% de precision')

In [ ]:
# Celda 6: Matriz de confusion
# Muestra cuantas veces el modelo predijo correctamente cada letra
from sklearn.metrics import confusion_matrix, classification_report

y_pred = mejor_modelo.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=le.classes_,
            yticklabels=le.classes_,
            cmap='Blues')
plt.title(f'Matriz de Confusion — {mejor_nombre}')
plt.xlabel('Prediccion del modelo')
plt.ylabel('Letra real')
plt.tight_layout()
plt.show()

print('Reporte detallado por letra:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Celda 7: Ejemplo de prediccion manual
# Se ingresan los 6 angulos de una sena y el modelo predice la letra

ejemplos = {
    'A': [8, 1, 1, 5, 165, 161],
    'B': [172, 174, 178, 171, 116, 96],
    'I': [6, 6, 179, 174, 163, 105],
}

print('Predicciones de ejemplo:')
print('-' * 40)
for letra_real, angulos in ejemplos.items():
    entrada = np.array(angulos).reshape(1, -1)
    pred_enc = mejor_modelo.predict(entrada)[0]
    pred_letra = le.inverse_transform([pred_enc])[0]
    correcto = 'correcto' if pred_letra == letra_real else 'incorrecto'
    print(f'Angulos {angulos} -> Prediccion: {pred_letra} ({correcto})')

## Conclusiones

- El modelo clasifica correctamente las 15 letras implementadas del abecedario LSM.
- El uso de angulos en lugar de coordenadas absolutas hace al sistema robusto ante cambios de posicion de la mano.
- La letra J se detecta mediante movimiento del menique, combinando vision artificial con logica dinamica.
- El componente de PLN acumula las letras predichas para formar palabras completas en tiempo real.